# A hybrid commit-message classifier

This notebook uses the same 339 commit titles from 10 skills. It checks `fix:`, `feat:`, and `refactor:` first. When none of these prefixes is present, it uses the broader keywords from the [*When Code Smells Meet ML* replication package](https://figshare.com/articles/dataset/_b_When_Code_Smells_Meet_ML_On_the_Lifecycle_of_ML-specific_Code_Smells_in_ML-enabled_Systems_b_-_Replication_Package/28167065).

In [3]:
from pathlib import Path
import pandas as pd

file = Path("../data/gitskills/commit_messages.json")
if not file.exists():
    file = Path("data/gitskills/commit_messages.json")

commits = pd.read_json(file)

## Rules

The keywords below are from `CodeSmileAnalysis/commit_message_classification.py` in version 3 of the replication package. The prefix rule is our addition. A recognized prefix takes priority over keywords; without one, a title can receive several labels.

In [4]:
prefixes = {"fix": "bug fixing", "feat": "new feature", "refactor": "refactoring"}

keywords = {
    "new feature": {"new", "feature", "feat:", "add", "creat", "introduc", "implement"},
    "bug fixing": {"fix", "repair", "error", "bug", "issue", "exception"},
    "enhancement": {"updat", "chang", "switch", "optimiz", "improv", "edit", "compatibl", "adapt", "updt", "allow", "revise", "modif", "upgrad", "export", "remov", "integrat", "support", "enhanc", "replac", "includ", "expos", "generat", "migrat"},
    "refactoring": {"renam", "restructur", "decoupl", "reorganiz", "rewrit", "rm", "refact", "rework", "ren:", "clean", "polish", "mov", "extract", "reorder", "re-order", "merg"}
}

def classify(subject):
    text = subject.lower()
    prefix = text.split(":", 1)[0].split("(", 1)[0].rstrip("!")
    if ":" in text and prefix in prefixes:
        return [prefixes[prefix]], "prefix"

    labels = [name for name, terms in keywords.items() if any(term in text for term in terms)]
    return labels, "keywords" if labels else "no match"

commits[["labels", "rule"]] = pd.DataFrame(
    commits["subject"].apply(classify).tolist(), index=commits.index
)

## Results

First, count which rule classified each title. Then compare the number of labels with keyword matching alone.

In [5]:
commits["rule"].value_counts().rename("commits").to_frame()

,commits
rule,
prefix,179
keywords,117
no match,43


In [4]:
commits["keywords_only"] = commits["subject"].str.lower().apply(
    lambda text: [name for name, terms in keywords.items() if any(term in text for term in terms)]
)

rows = []
for name, column in [("Keywords only", "keywords_only"), ("Hybrid", "labels")]:
    sizes = commits[column].str.len()
    rows.append([name, (sizes == 0).sum(), (sizes == 1).sum(), (sizes > 1).sum()])

pd.DataFrame(rows, columns=["method", "no label", "one label", "multiple labels"]).set_index("method")

,no label,one label,multiple labels
method,,,
Keywords only,60,160,119
Hybrid,43,263,33


## Examples

`fix: address` is classified by its prefix. A title without a recognized prefix still uses substring matching, so `add` can match inside `address`.

In [5]:
examples = commits[commits["subject"].str.contains("address PR", case=False)]
for _, row in examples.iterrows():
    print(row["subject"])
    print("Labels:", row["labels"], "| Rule:", row["rule"], "\n")

fix: address PR review feedback
Labels: ['bug fixing'] | Rule: prefix 

fix: [T703] address PR #151 review — gate conditional pushes to their guard in the drift oracle; sync dispatchSpec across adapter arg-list docs
Labels: ['bug fixing'] | Rule: prefix 

fix: [T622] address PR #174 review — restore before fix rounds, path-scope workspace base reads
Labels: ['bug fixing'] | Rule: prefix 

test: [T706] address PR #186 review — pin reviewer map (AC7) + drive P-SQ control live
Labels: ['new feature'] | Rule: keywords 



## What this tells us

The prefix rule prevents some misleading extra labels, but the keyword fallback still needs checking. These are labels for commit titles, not verified descriptions of the `SKILL.md` changes.